In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import os
from model.expression_embedding import expression_GVAE
from common.data_processing import exp_data_preprocessing
from common.dataset import CLIPDataset
from scstGCN.align_image_pos import align_image,align_pos
from scstGCN.histological_feature_map import his_ex
from scstGCN.RGB_feature_map import rgb_ex
from scstGCN.pos_feature_map import pos_ex
from scstGCN.impute import impute_internel

input_path='./data/patches16_V_HD_Small_Intestine_bin16/'
names = [d for d in os.listdir(input_path) if os.path.isdir(os.path.join(input_path, d))]

adata_list, HE_image_paths, nuclei_mask_paths=exp_data_preprocessing(input_path)

for i in range(len(adata_list)):
    sc.pp.filter_genes(adata_list[i], min_cells=1)
    sc.pp.filter_cells(adata_list[i], min_genes=1)
# 计算所有anndata共享的var_names
shared_var_names = set(adata_list[0].var_names)
for adata in adata_list[1:]:
    shared_var_names.intersection_update(adata.var_names)
print(len(shared_var_names))

for i in range(len(adata_list)):
    adata_list[i] = adata_list[i][:, list(shared_var_names)]
    
print(adata_list[0])

Input_Method='hvg'
new_adata_list=expression_GVAE(adatas=adata_list,method='GATE',feature_method=Input_Method,n_top_genes=1000)
print(new_adata_list[0])

In [ ]:
from PIL import Image
image=Image.open(HE_image_paths[0])
image = np.array(image)
image = image[:,:,0:3]
block_r=adata_list[0].uns['block_r']


In [ ]:
def get_locs(locs,wsi,target_shape=None):

    # change xy coordinates to ij coordinates
    locs = np.stack([locs['y'], locs['x']], -1)

    # match coordinates of embeddings and spot locations
    if target_shape is not None:
        current_shape = np.array(wsi.shape[:2])
        rescale_factor = current_shape // target_shape
        locs = locs.astype(float)
        locs /= rescale_factor
    locs = locs.round().astype(int)
    return locs
def get_disk_mask(radius, boundary_width=None):
    radius_ceil = np.array(radius).astype(int)
    locs = np.meshgrid(
            np.arange(-radius_ceil, radius_ceil+1),
            np.arange(-radius_ceil, radius_ceil+1),
            indexing='ij')
    locs = np.stack(locs, -1)
    distsq = (locs**2).sum(-1)
    isin = distsq <= radius**2
    if boundary_width is not None:
        isin *= distsq >= (radius-boundary_width)**2
    return isin

def get_patches_flat(img, locs, mask):
    shape = np.array(mask.shape)
    mask = np.ones_like(mask, dtype=bool)
    center = shape // 2
    r = np.stack([-center, shape-center], -1)  # offset
    x_list = []
    for s in locs:
        patch = img[
                s[0]+r[0][0]:s[0]+r[0][1],
                s[1]+r[1][0]:s[1]+r[1][1]]
        x = patch[mask]
        x_list.append(x)
    x_list = np.stack(x_list)
    return x_list
def processing_locs(adata,image,final_embs):
    locs=np.array(adata.obsm['spatial'])
    locs=pd.DataFrame(locs,index=adata.obs_names,columns=['x','y'])
    locs=align_pos(locs)
    final_locs=get_locs(locs,image,final_embs.shape[:2])   
    return final_locs 

In [ ]:

device='cuda:2'
img=align_image(image)
img_emb = his_ex(img, device)
rgb_emb = rgb_ex(img)
pos_emb = pos_ex(img_emb)
embs = dict(his=img_emb, rgb=rgb_emb, pos=pos_emb)

final_embs=np.concatenate([embs['his'], embs['rgb'], embs['pos']]).transpose(1, 2, 0)

In [ ]:
train_index=[i for i in range(len(new_adata_list))]

for i in train_index: 
    test_index=[i]
    new_train_index = [item for item in train_index if item not in test_index]
    train_adata_list = [new_adata_list[j] for j in new_train_index]
    adata=train_adata_list[0].concatenate(*train_adata_list[1:], join='inner')
    adata.uns['block_r']=block_r
    
    test_adata=new_adata_list[i]
    test_adata.uns['block_r']=block_r
    
    factor = 16
    ori_radius=adata.uns['block_r']
    radius = ori_radius / factor
    cnts=adata.to_df()
    
    final_train_locs=processing_locs(adata,image,final_embs)
    final_test_locs=processing_locs(test_adata,image,final_embs)
    save_path=os.path.join(input_path,names[test_index[0]],'results/')    
    n_train = cnts.shape[1]
    batch_size = min(128, n_train//16)    
    
    y_grp=impute_internel(embs=final_embs, cnts=cnts, locs=final_train_locs, radius=radius,
        ori_radius=ori_radius, epochs=300, batch_size=batch_size,
        n_states=5, prefix=save_path,
        load_saved=False, n_jobs=1)
    mask = get_disk_mask(radius)
    x = get_patches_flat(y_grp, final_test_locs, mask)

    predict_adata=test_adata.copy()
    predict_adata.X = x.mean(1)
    predict_adata.write_h5ad(os.path.join(save_path, 'scstGCN_pre.h5ad'))
    test_adata.write_h5ad(os.path.join(save_path, 'scstGCN_gt.h5ad'))
    